# 02c — Train `tomato_ripeness_v1` với augmentation màu/ánh sáng tăng cường

**Bối cảnh:** `02b_review_domain_gap_ripeness.ipynb` xác nhận domain gap trên `test_outdomain_openfield` là **domain shift thật** (không phải lỗi nhãn), với 2 nguyên nhân độc lập:
1. Lệch màu sắc/ánh sáng (chênh saturation 44,5/255 giữa trong miền và ngoài miền) — gây thiên lệch phân loại độ chín có hệ thống (93% lỗi đánh giá quả *xanh hơn* thực tế).
2. Nền đất/đá ngoài đồng chưa từng xuất hiện lúc train — gây báo thừa hàng loạt (3.231 box thừa / 2.802 box GT thật).

## Phạm vi notebook này (đọc kỹ trước khi kỳ vọng kết quả)
Notebook này **chỉ xử lý nguyên nhân (1)** bằng cách tăng cường augmentation màu/ánh sáng khi train lại (`hsv_s`, `hsv_v`, `hsv_h` cao hơn mặc định).

**Không xử lý nguyên nhân (2)** trong notebook này: cách xử lý đúng là bổ sung ảnh nền không chứa quả (hard negative) làm quen với nền đất/ngoài trời — nhưng nguồn ảnh nền đó chỉ có trong `test_outdomain_openfield`, và dự án đã chủ động cách ly tập này khỏi train ngay từ đầu (`01_build_tomato_ripeness_v1.ipynb`, license `openfield_bd` chưa xác nhận + giữ tính độc lập của test set) — dùng nó để tạo hard negative sẽ vi phạm chính nguyên tắc đó và làm mất giá trị đánh giá độc lập. Xử lý đúng nguyên nhân (2) cần ảnh nền thật mới (từ camera IMX179 hoặc nguồn public khác chưa có trong dự án) — để lại cho khi có dữ liệu đó.

Vì vậy, **kỳ vọng hợp lý**: cải thiện (nếu có) chỉ nên thấy rõ ở phần nhầm lẫn giữa các mức độ chín (nguyên nhân 1), còn số box báo thừa trên nền lạ (nguyên nhân 2) nhiều khả năng gần như không đổi. Notebook đo cả hai để biết chính xác, không suy đoán.

## Trước khi chạy
1. **Add Input** → output đã Save Version của `01_build_tomato_ripeness_v1.ipynb` (`tomato_ripeness_v1`).
2. Accelerator: **GPU** (P100/T4x2) bắt buộc. Internet: ON.

## Cấu hình
Giống hệt `02_train_ripeness_baseline.ipynb` (YOLOv8n, imgsz 640, epochs 100, batch 16, patience 20, seed 42) — chỉ khác augmentation màu:

| Tham số | Mặc định ultralytics | Notebook này |
|---|---|---|
| `hsv_h` (hue) | 0.015 | 0.02 |
| `hsv_s` (saturation) | 0.7 | 0.9 |
| `hsv_v` (brightness) | 0.4 | 0.6 |

Đánh giá lại trên cả `test` và `test_outdomain_openfield`, so trực tiếp với số liệu baseline gốc (đã lưu cứng trong notebook để so sánh, không cần chạy lại `02_train_ripeness_baseline.ipynb`).

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=False)

import torch

print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "Không có GPU. Vào Settings (panel phải) -> Accelerator -> chọn GPU T4 x2 hoặc P100, "
        "rồi chạy lại từ đầu."
    )

### Bước 1 — Tự nhận diện dataset + vá `data.yaml`

In [ ]:
from pathlib import Path

import yaml

TARGET_CLASSES = ["fruit_green_unripe", "fruit_turning", "fruit_ripe"]


def find_dataset_root():
    for yf in Path("/kaggle/input").rglob("data.yaml"):
        try:
            y = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        names = y.get("names") if isinstance(y, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names == TARGET_CLASSES:
            return yf.parent
    return None


DATASET_ROOT = find_dataset_root()
if DATASET_ROOT is None:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy data.yaml khớp 3 class của tomato_ripeness_v1 trong /kaggle/input."
    )

DATA_YAML = DATASET_ROOT / "data.yaml"
OUTDOMAIN_DIR = DATASET_ROOT / "test_outdomain_openfield"
print("DATASET_ROOT:", DATASET_ROOT)

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

orig_yaml = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
fixed_yaml = {
    "path": str(DATASET_ROOT),
    "train": orig_yaml.get("train", "train/images"),
    "val": orig_yaml.get("val", "val/images"),
    "test": orig_yaml.get("test", "test/images"),
    "names": orig_yaml["names"],
}
FIXED_DATA_YAML = WORKING / "data_ripeness.yaml"
with open(FIXED_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_yaml, f, allow_unicode=True, sort_keys=False)
print("data.yaml đã vá:", FIXED_DATA_YAML)

for split_key in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / fixed_yaml[split_key]
    n = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    print(f"  {split_key:6s}: {n} ảnh")
    if n == 0:
        raise FileNotFoundError(f"{split_key} rỗng tại {img_dir}.")

### Bước 2 — Train YOLOv8n với augmentation màu tăng cường

In [ ]:
from ultralytics import YOLO

RUN_NAME = "tomato_ripeness_v1_yolov8n_augmented"
RUNS_DIR = WORKING / "runs"

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(FIXED_DATA_YAML),
    imgsz=640,
    epochs=100,
    batch=16,
    patience=20,
    seed=42,
    pretrained=True,
    hsv_h=0.02,
    hsv_s=0.9,
    hsv_v=0.6,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
print("\nbest.pt:", BEST_PT, "tồn tại:", BEST_PT.exists())

### Bước 3 — Đánh giá trên `test` và `test_outdomain_openfield`

In [ ]:
best_model = YOLO(str(BEST_PT))

test_metrics = best_model.val(
    data=str(FIXED_DATA_YAML), split="test", imgsz=640,
    project=str(RUNS_DIR), name=f"{RUN_NAME}_eval_test", exist_ok=True,
)
print("== test (cùng miền) ==")
print(f"Precision={test_metrics.box.mp:.4f}  Recall={test_metrics.box.mr:.4f}  "
      f"mAP50={test_metrics.box.map50:.4f}  mAP50-95={test_metrics.box.map:.4f}")
for i, cname in enumerate(TARGET_CLASSES):
    print(f"  {cname:20s} P={test_metrics.box.p[i]:.4f}  R={test_metrics.box.r[i]:.4f}  "
          f"AP50={test_metrics.box.ap50[i]:.4f}")

outdomain_metrics = None
if OUTDOMAIN_DIR.exists() and any((OUTDOMAIN_DIR / "images").glob("*")):
    outdomain_yaml = {
        "path": str(DATASET_ROOT),
        "train": fixed_yaml["train"],
        "val": "test_outdomain_openfield/images",
        "names": fixed_yaml["names"],
    }
    OUTDOMAIN_DATA_YAML = WORKING / "data_ripeness_outdomain.yaml"
    with open(OUTDOMAIN_DATA_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(outdomain_yaml, f, allow_unicode=True, sort_keys=False)

    outdomain_metrics = best_model.val(
        data=str(OUTDOMAIN_DATA_YAML), split="val", imgsz=640,
        project=str(RUNS_DIR), name=f"{RUN_NAME}_eval_outdomain", exist_ok=True,
    )
    print("\n== test_outdomain_openfield (ngoài miền) ==")
    print(f"Precision={outdomain_metrics.box.mp:.4f}  Recall={outdomain_metrics.box.mr:.4f}  "
          f"mAP50={outdomain_metrics.box.map50:.4f}  mAP50-95={outdomain_metrics.box.map:.4f}")
    for i, cname in enumerate(TARGET_CLASSES):
        print(f"  {cname:20s} P={outdomain_metrics.box.p[i]:.4f}  R={outdomain_metrics.box.r[i]:.4f}  "
              f"AP50={outdomain_metrics.box.ap50[i]:.4f}")

### Bước 4 — So sánh trực tiếp với baseline gốc
Số liệu baseline gốc (từ `02_train_ripeness_baseline.ipynb`, đã lưu cứng vì đây là notebook độc lập không đọc lại được run trước trên Kaggle).

In [ ]:
import pandas as pd

BASELINE = {
    "test":      {"precision": 0.8259, "recall": 0.8265, "map50": 0.8784, "map50_95": 0.6941},
    "outdomain": {"precision": 0.5626, "recall": 0.5593, "map50": 0.4731, "map50_95": 0.4296},
}

rows = [{
    "eval_set": "test (cùng miền)",
    "precision": round(float(test_metrics.box.mp), 4), "recall": round(float(test_metrics.box.mr), 4),
    "map50": round(float(test_metrics.box.map50), 4), "map50_95": round(float(test_metrics.box.map), 4),
    "baseline_map50": BASELINE["test"]["map50"],
    "delta_map50": round(float(test_metrics.box.map50) - BASELINE["test"]["map50"], 4),
}]
if outdomain_metrics is not None:
    rows.append({
        "eval_set": "test_outdomain_openfield",
        "precision": round(float(outdomain_metrics.box.mp), 4), "recall": round(float(outdomain_metrics.box.mr), 4),
        "map50": round(float(outdomain_metrics.box.map50), 4), "map50_95": round(float(outdomain_metrics.box.map), 4),
        "baseline_map50": BASELINE["outdomain"]["map50"],
        "delta_map50": round(float(outdomain_metrics.box.map50) - BASELINE["outdomain"]["map50"], 4),
    })

compare_df = pd.DataFrame(rows)
print(compare_df.to_string(index=False))

if outdomain_metrics is not None:
    delta = rows[1]["delta_map50"]
    if delta > 0.02:
        print(f"\n[KẾT QUẢ] mAP@0.5 trên outdomain tăng {delta:+.4f} — augmentation màu có giúp ích thật.")
    elif delta < -0.02:
        print(f"\n[KẾT QUẢ] mAP@0.5 trên outdomain giảm {delta:+.4f} — augmentation mạnh hơn không giúp, "
              "thậm chí có thể làm nhiễu tín hiệu màu quan trọng để phân biệt độ chín. Nên quay lại cấu hình baseline.")
    else:
        print(f"\n[KẾT QUẢ] mAP@0.5 trên outdomain gần như không đổi ({delta:+.4f}) — củng cố thêm giả thuyết "
              "nguyên nhân chính là báo thừa trên nền lạ (nguyên nhân 2), thứ mà augmentation màu không giải quyết được.")

### Bước 5 — Đối chiếu chi tiết theo IoU (giống `02b`)
Kiểm tra đúng giả thuyết đã nêu ở đầu: tỷ lệ nhầm lẫn (misclassified) có giảm không, còn số box báo thừa (extra, do nền lạ) có đổi không.

In [ ]:
def yolo_txt_to_xyxy(label_path, w, h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        xmin, ymin = (xc - bw / 2) * w, (yc - bh / 2) * h
        xmax, ymax = (xc + bw / 2) * w, (yc + bh / 2) * h
        boxes.append((cid, xmin, ymin, xmax, ymax))
    return boxes


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


IOU_THRESH = 0.5
BASELINE_ERR = {"correct": 1762, "misclassified": 835, "missed": 205, "extra": 3231, "n_gt": 2802}

if OUTDOMAIN_DIR.exists():
    from PIL import Image

    images_dir = OUTDOMAIN_DIR / "images"
    labels_dir = OUTDOMAIN_DIR / "labels"
    all_imgs = sorted(images_dir.glob("*"))

    n_correct = n_miscls = n_missed = n_extra = n_gt_total = 0
    under_ripe_errors = 0  # true class "chin hon" -> doan "xanh hon" (vd turning->green, ripe->turning/green)
    over_ripe_errors = 0

    BATCH = 32
    for i in range(0, len(all_imgs), BATCH):
        batch = all_imgs[i:i + BATCH]
        results = best_model.predict(source=[str(p) for p in batch], imgsz=640, conf=0.25, verbose=False)
        for img_path, res in zip(batch, results):
            with Image.open(img_path) as im:
                w, h = im.size
            gt_boxes = yolo_txt_to_xyxy(labels_dir / (img_path.stem + ".txt"), w, h)
            pred_xyxy = res.boxes.xyxy.cpu().numpy().tolist() if len(res.boxes) else []
            pred_cls = res.boxes.cls.cpu().numpy().astype(int).tolist() if len(res.boxes) else []
            preds = list(zip(pred_cls, pred_xyxy))

            used_pred = set()
            for gt_cid, *gt_box in gt_boxes:
                n_gt_total += 1
                best_iou, best_j = 0.0, -1
                for j, (pc, pbox) in enumerate(preds):
                    if j in used_pred:
                        continue
                    iou = iou_xyxy(tuple(gt_box), tuple(pbox))
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                if best_iou >= IOU_THRESH:
                    used_pred.add(best_j)
                    pc = preds[best_j][0]
                    if pc == gt_cid:
                        n_correct += 1
                    else:
                        n_miscls += 1
                        if pc < gt_cid:
                            under_ripe_errors += 1  # doan "xanh hon" nhan that
                        else:
                            over_ripe_errors += 1
                else:
                    n_missed += 1
            n_extra += len(preds) - len(used_pred)
        print(f"  đã xử lý {min(i + BATCH, len(all_imgs))}/{len(all_imgs)} ảnh...")

    print(f"\n== Kết quả notebook này (augmentation màu tăng cường) ==")
    print(f"correct={n_correct}  misclassified={n_miscls}  missed={n_missed}  extra={n_extra}  n_gt={n_gt_total}")
    print(f"  trong đó nhầm 'xanh hơn thực tế'={under_ripe_errors} | nhầm 'chín hơn thực tế'={over_ripe_errors}")

    print(f"\n== So với 02b (baseline gốc) ==")
    print(f"correct: {BASELINE_ERR['correct']} -> {n_correct}")
    print(f"misclassified: {BASELINE_ERR['misclassified']} -> {n_miscls}")
    print(f"missed: {BASELINE_ERR['missed']} -> {n_missed}")
    print(f"extra (báo thừa): {BASELINE_ERR['extra']} -> {n_extra}")

### Bước 6 — Lưu model + báo cáo so sánh

In [ ]:
import shutil

MODELS_DIR = WORKING / "models"
REPORTS_DIR = WORKING / "reports" / RUN_NAME
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_PT, MODELS_DIR / "tomato_ripeness_yolov8n_augmented.pt")
compare_df.to_csv(REPORTS_DIR / "metrics_vs_baseline.csv", index=False)

error_compare = pd.DataFrame([
    {"metric": "correct", "baseline": BASELINE_ERR["correct"], "augmented": n_correct},
    {"metric": "misclassified", "baseline": BASELINE_ERR["misclassified"], "augmented": n_miscls},
    {"metric": "missed", "baseline": BASELINE_ERR["missed"], "augmented": n_missed},
    {"metric": "extra_bao_thua", "baseline": BASELINE_ERR["extra"], "augmented": n_extra},
])
error_compare.to_csv(REPORTS_DIR / "error_breakdown_vs_baseline.csv", index=False)

train_config = {
    "model": "yolov8n.pt", "imgsz": 640, "epochs": 100, "batch": 16, "patience": 20, "seed": 42,
    "pretrained": True, "hsv_h": 0.02, "hsv_s": 0.9, "hsv_v": 0.6,
    "note": "Tang cuong augmentation mau/anh sang so voi baseline (hsv_s 0.7->0.9, hsv_v 0.4->0.6, hsv_h 0.015->0.02). "
            "Khong xu ly nguyen nhan bao thua tren nen la (can du lieu that).",
}
with open(REPORTS_DIR / "train_config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(train_config, f, allow_unicode=True, sort_keys=False)

print("Đã lưu:")
print(" -", MODELS_DIR / "tomato_ripeness_yolov8n_augmented.pt")
print(" -", REPORTS_DIR / "metrics_vs_baseline.csv")
print(" -", REPORTS_DIR / "error_breakdown_vs_baseline.csv")
print(" -", REPORTS_DIR / "train_config.yaml")

## Kết quả & cách đọc
- Nếu `misclassified` giảm rõ và đặc biệt `under_ripe_errors` (nhầm "xanh hơn thực tế") giảm mạnh so với baseline (835 / phần lớn là 777 lỗi under-ripe) → augmentation màu có tác dụng thật với nguyên nhân (1), nên cân nhắc dùng bản này thay baseline.
- Nếu `extra` (báo thừa) không đổi nhiều so với baseline (3.231) → đúng như dự đoán, augmentation màu không giải quyết được nguyên nhân (2) — cần dữ liệu nền thật, không phải augmentation.
- Nếu cả `test` (cùng miền) cũng giảm điểm so với baseline gốc (0,878) → augmentation quá mạnh đang làm nhiễu tín hiệu màu cần thiết để phân biệt độ chín ngay cả trong điều kiện bình thường — nên giảm bớt `hsv_s`/`hsv_v` hoặc quay lại baseline.

## Bước tiếp theo
- So sánh xong, cập nhật `ai/README.md` với kết luận: augmentation màu có ích tới đâu, và xác nhận lại rằng nguyên nhân báo thừa trên nền lạ vẫn cần dữ liệu thật để giải quyết (mục 1 trong "Đề xuất bước tiếp theo").
- Nếu bản augmented tốt hơn rõ rệt trên `test_outdomain_openfield` mà không đánh đổi hiệu năng trên `test`, có thể thay thế `models/tomato_ripeness_yolov8n_baseline.pt` bằng bản này làm baseline chính thức mới.